# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print summary of dataset
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset DOI: {getattr(metadata,'identifier', 'N/A')}")
print(f"Date published: {getattr(metadata,'datePublished', 'N/A')}")
print(f"License: {getattr(metadata,'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references are by their `@id`.

Let's list all record sets, their IDs, fields, and their columns.

In [ ]:
record_sets = metadata.recordSet if getattr(metadata, 'recordSet', None) else []
if not record_sets:
    print("No record sets defined in metadata. The dataset may consist of a single main record set derived from tabular content or files in `distribution`.")

# If none, check for distribution DataDownloads
distributions = getattr(metadata, 'distribution', [])
for dist in distributions:
    print(f"Distribution @id: {dist.get('@id', '')} -- {dist.get('name', '')}")
    # The columns or main tabular file is usually linked per distribution
    file_object = dist.get('encoding', [{}])[0] if 'encoding' in dist and len(dist['encoding']) else None
    if file_object:
        print(f"  FileObject columns: {file_object.get('column', [])}")

# For mlcroissant, use dataset.record_sets attribute
print("\nDiscovered Record sets and IDs:")
for rset in dataset.record_sets:
    print(f"  RecordSet: {rset['@id']}")
    print(f"    Name: {rset.get('name', '')}")
    print(f"    Description: {rset.get('description', '')}")
    fields = rset.get('field', [])
    print(f"    Field @ids: {[f['@id'] for f in fields]}")
    for field in fields:
        print(f"      Field: {field['@id']} - {field.get('name','')}, dataType: {field.get('dataType', '')}, column: {field.get('column', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If the dataset contains exactly one record set, we use that; otherwise, you can manually select the record set `@id` of interest.

In [ ]:
# Find all available record_set IDs in the dataset croissant file
record_set_ids = [rset['@id'] for rset in dataset.record_sets]
print("Record set @ids found:", record_set_ids)

# If none found, use the main distribution as default record set (common for data-centric Croissant schemas without explicit record sets)
if not record_set_ids:
    from pprint import pprint
    print("No explicit record sets; attempting to extract tabular data from main distribution.")
    # Try extracting the inferred default record set (usually the tabular file in distribution)
    records = list(dataset.records())
    df = pd.DataFrame(records)
    print("Columns:", df.columns.tolist())
    print(df.head())
    dataframes = {'default': df}
    used_record_set = 'default'
else:
    dataframes = {}
    for rset_id in record_set_ids:
        records = list(dataset.records(record_set=rset_id))
        dataframes[rset_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet: {rset_id}, Shape: {dataframes[rset_id].shape}")
    # Select the first for demonstration
    used_record_set = record_set_ids[0]
    print(f"Columns in {used_record_set}:", dataframes[used_record_set].columns.tolist())
    print(dataframes[used_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on numeric fields, normalization, removing outliers, or grouping data to prepare for further analysis.

We'll:
- Select a numeric field (e.g., protein abundance, coverage)
- Filter records with high abundance/coverage
- Normalize that field
- Group by a protein attribute (if available)

Please reference fields by their `@id` (column names).

In [ ]:
df = dataframes[used_record_set]
# Detect likely numeric fields
candidates = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
print(f"Possible numeric fields: {candidates}")
if not candidates:
    # Try force-convert some expected columns
    import numpy as np
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except:
            pass
    candidates = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    print(f"Detected numeric columns after type inference: {candidates}")

# Choose a numeric field ('Coverage' or 'Abundance') if present, otherwise take the first
if candidates:
    if 'Coverage' in candidates:
        numeric_field = 'Coverage'
    elif 'Abundance' in candidates:
        numeric_field = 'Abundance'
    else:
        numeric_field = candidates[0]
else:
    raise Exception('No numeric field found for EDA.')

print(f"Selected numeric field for EDA: {numeric_field}")
threshold = df[numeric_field].mean() if df[numeric_field].dtype != 'O' else 0   # safeguard

# Filter by numeric field
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a key attribute, e.g., 'Gene', 'Protein', or the first object (non-numeric, non-index) field
group_fields = [col for col in df.columns if df[col].dtype == 'object']
if group_fields:
    group_field = group_fields[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (showing top rows):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the distribution of the numeric field selected for EDA, and a scatter plot if another numeric is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# Scatter plot of top two numeric fields if available
if len(candidates) > 1:
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=df[candidates[0]], y=df[candidates[1]])
    plt.xlabel(candidates[0])
    plt.ylabel(candidates[1])
    plt.title(f"{candidates[0]} vs {candidates[1]}")
    plt.show()

## 6. Conclusion
We have demonstrated how to load, examine, and process the FAIR\u2072 mass spectrometry dataset using `mlcroissant`. Dataset metadata and tabular records are accessible via Croissant `@id` structure, facilitating consistent, reproducible analyses.

**Key findings:**
- The dataset contains rich protein abundance and sequence information, suitable for proteomic bioinformatics exploration.
- Outlier and high-value filtering, field normalization, and grouped summaries allow for biological insight extraction.
- Workflow is reproducible and agnostic to downstream processing thanks to the Croissant standard.

You can now extend this notebook to deeper analysis or machine learning tasks!